# LC #739 — Daily Temperatures
**Category:** Monotonic Stack | **Difficulty:** Medium | **Day 3**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>The Problem:</strong> Given array <code>temperatures</code>, for each day return how many
days until a warmer temperature. Return 0 if no warmer day exists.
</div>

**Example:**
```
Input:  [73, 74, 75, 71, 69, 72, 76, 73]
Output: [ 1,  1,  4,  2,  1,  1,  0,  0]
```
*Day 2 (75°): next warmer day is day 6 (76°) — 4 days away.*

**Core Insight:** Maintain a **monotonic decreasing stack** of indices of "unresolved" days. When a warmer temperature arrives, pop and compute the gap.

## Mental Models

**1. Unresolved Days Waiting for Resolution**
Each index on the stack represents a day that hasn't yet found a warmer day. The stack holds them "in suspense." When a warmer day arrives, it resolves all waiting days that are colder than it.

**2. Monotonic Decreasing = Never Redundant**
We maintain the stack in decreasing order of temperatures. Why? Because if temperature[i] ≤ temperature[j] and i < j, then i will never be the answer for any day after j — j always blocks it. So i is useless once j is on the stack.

**3. Amortized O(n)**
Each element enters the stack once and exits at most once. Despite the inner while loop, total operations ≤ 2n across all iterations.

In [ ]:
# Brute Force — O(n²) time
def dailyTemperatures_brute(temperatures: list[int]) -> list[int]:
    result = [0] * len(temperatures)
    for i in range(len(temperatures)):
        for j in range(i + 1, len(temperatures)):
            if temperatures[j] > temperatures[i]:
                result[i] = j - i
                break
    return result

temps = [73, 74, 75, 71, 69, 72, 76, 73]
print(dailyTemperatures_brute(temps))   # [1, 1, 4, 2, 1, 1, 0, 0]

## Stack Trace: `[73, 74, 75, 71, 69, 72, 76, 73]`

| i | temp | While loop (pop warmer) | stack (indices) | result[popped] |
|---|------|------------------------|-----------------|----------------|
| 0 | 73 | none | [0] | — |
| 1 | 74 | pop 0 (73<74) | [1] | result[0]=1-0=1 |
| 2 | 75 | pop 1 (74<75) | [2] | result[1]=2-1=1 |
| 3 | 71 | none (71<75) | [2,3] | — |
| 4 | 69 | none (69<71) | [2,3,4] | — |
| 5 | 72 | pop 4 (69<72), pop 3 (71<72) | [2,5] | result[4]=1, result[3]=2 |
| 6 | 76 | pop 5 (72<76), pop 2 (75<76) | [6] | result[5]=1, result[2]=4 |
| 7 | 73 | none (73<76) | [6,7] | — |
| end | — | stack [6,7] → result stays 0 | | — |

In [ ]:
# Optimal — O(n) time, O(n) space (amortized)
def dailyTemperatures(temperatures: list[int]) -> list[int]:
    result = [0] * len(temperatures)
    stack = []   # indices of unresolved days (temperatures are decreasing)

    for i, temp in enumerate(temperatures):
        # Current day is warmer — resolve all colder waiting days
        while stack and temperatures[stack[-1]] < temp:
            prev_day = stack.pop()
            result[prev_day] = i - prev_day

        stack.append(i)   # current day is now waiting
    # Days still in stack: no warmer day found → result stays 0

    return result

# Test
temps = [73, 74, 75, 71, 69, 72, 76, 73]
print(dailyTemperatures(temps))   # [1, 1, 4, 2, 1, 1, 0, 0]

# Edge cases
print(dailyTemperatures([30, 40, 50, 60]))   # [1, 1, 1, 0]
print(dailyTemperatures([30, 60, 90]))        # [1, 1, 0]
print(dailyTemperatures([90, 80, 70, 60]))   # [0, 0, 0, 0]

## Complexity Analysis

| Approach | Time | Space |
|----------|------|-------|
| Brute Force | O(n²) | O(1) |
| **Monotonic Stack** | **O(n)** | **O(n)** |

**Why O(n) despite while loop?** Each index is pushed to the stack at most once and popped at most once. Total push + pop operations ≤ 2n across all iterations — amortized O(1) per element.

**Stack max size:** In the worst case (monotonically decreasing temperatures), all indices accumulate in the stack → O(n) space.

## Interview Q&A

**Q: Why maintain a monotonic decreasing stack?**
A: We only keep elements that could be the answer for future days. If temperature[i] < temperature[j] and i < j, then j is "blocking" i — any day warmer than i is also warmer than j, and j is closer. So i can never be the "next warmer" for anything to the right of j. We pop it.

**Q: What's amortized O(n)?**
A: Each element is pushed exactly once and popped at most once. The inner while loop's total iterations across all i = at most n pops. So total work = O(n) pushes + O(n) pops = O(n).

**Q: What about elements still in the stack at the end?**
A: They represent days with no warmer day to the right. Result stays 0 (initialized value). No cleanup needed.

**Q: Data engineering application?**
A: "Next time a metric exceeds its current value" — first-occurrence detection. "For each day's CPU reading, how many days until a higher reading?" Same algorithm.

## The Citi Angle

**"Next time CPU exceeds current peak" — capacity trend detection:**

```python
# For each day, how many days until CPU exceeds today's reading?
# Monotonic stack: same algorithm as dailyTemperatures

def days_until_higher_cpu(daily_readings: list[float]) -> list[int]:
    """For each day, days until a higher CPU reading is observed."""
    result = [0] * len(daily_readings)
    stack = []   # indices of unresolved days

    for i, cpu in enumerate(daily_readings):
        while stack and daily_readings[stack[-1]] < cpu:
            prev = stack.pop()
            result[prev] = i - prev
        stack.append(i)

    return result

# 8 days of srv-01 CPU readings
readings = [72.5, 68.1, 75.3, 71.0, 69.2, 77.8, 91.3, 88.4]
gaps = days_until_higher_cpu(readings)
for i, (cpu, gap) in enumerate(zip(readings, gaps)):
    note = f"→ exceeded in {gap} days" if gap else "→ no higher reading found"
    print(f"Day {i+1}: {cpu:.1f}% {note}")
```

**Interview tie-in:** "Daily temperatures is the 'next greater element' pattern. At Citi, I used this to find the first day a server's CPU exceeded its historical peak — an early warning signal for capacity planning."

## Summary

| | Value |
|--|--|
| **Pattern** | Monotonic Stack (decreasing) |
| **Time** | O(n) amortized |
| **Space** | O(n) |
| **Stack content** | Indices of days waiting for a warmer day |
| **Say in interview** | "Monotonic decreasing stack of unresolved indices. When warmer temp arrives, pop all colder days and compute gap. Amortized O(n): each element pushed and popped at most once." |